# Tome 14, Chapter 7: Selberg's central limit theorem (Python companion)

Exercises first, solutions at the end. We build a fast evaluator of $\zeta$ on the critical line (the Riemann–Siegel formula), look at the law of $\log|\zeta(\frac12+it)|$ at two heights, test the Gaussian moments of the prime sum, measure the cost of moving off the line, and check the twisted second moment numerically. The computer science application: **the Riemann–Siegel formula**, which evaluates $\zeta(\frac12+it)$ with about $\sqrt{t/2\pi}$ terms instead of $t$, the algorithm behind every large computation of zeros. Requires `numpy`, `scipy` and `mpmath`.

In [ ]:
import math, time
import numpy as np
import mpmath
from scipy.stats import kstest, norm
rng = np.random.default_rng(7)

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

## Exercises

**P7.1 (★) The Riemann–Siegel formula.** With $\theta(t)=\frac t2\log\frac t{2\pi}-\frac t2-\frac\pi8+\frac1{48t}+\frac7{5760t^3}$, $a=\sqrt{t/2\pi}$, $N=\lfloor a\rfloor$ and $p=a-N$,
$$Z(t)=2\sum_{n\le N}\frac{\cos(\theta(t)-t\log n)}{\sqrt n}+(-1)^{N-1}a^{-1/2}\,\frac{\cos 2\pi(p^2-p-\frac1{16})}{\cos2\pi p}+O(t^{-3/4}),$$
and $|Z(t)|=|\zeta(\frac12+it)|$. Implement it with numpy for an array of $t$, compare with `mpmath.siegelz` at a few points, and time it on $10^4$ points near $t=10^6$.

**P7.2 (★★) Selberg's law at two heights.** Sample $\log|\zeta(\frac12+it)|$ at $20\,000$ random $t\in[10^6,2\cdot10^6]$ and $5000$ random $t\in[10^{10},2\cdot10^{10}]$. Compare the sample variance with $\frac12\log\log T$ and with the random matrix value $\frac12(\log N+\gamma+1)$, $N=\log\frac{1.5T}{2\pi}$, and run Kolmogorov–Smirnov tests against both Gaussians (Theorem 7.1, Figure 7.1).

**P7.3 (★★) Moments of the prime sum.** With $T=10^6$, $\sigma_0=\frac12+\frac1{\log T}$ and $X=100$, compute the normalised moments $\mathbb E_T[(\operatorname{Re}P_0)^k]/(v/2)^{k/2}$, $k=1,\dots,6$, on $10^5$ random $t$, and compare with $0,1,0,3,0,15$ (Proposition 7.7). Explain why the low moments agree although $v$ is small, and why the sixth does not.

**P7.4 (★★) The cost of leaving the line.** On $400$ random $t\in[10^6,2\cdot10^6]$, compute the mean of $|\log|\zeta(\frac12+it)|-\log|\zeta(\sigma+it)||$ for $\sigma-\frac12=\frac c{\log T}$, $c\in\{0.25,0.5,1,2,4\}$, and compare with $c$ (Proposition 7.3 predicts at most a constant times $c$).

**P7.5 (★★★) The twisted second moment.** For $T=1000$ and $\sigma=0.6$, compute $\int_T^{2T}(h/k)^{it}|\zeta(\sigma+it)|^2\dd t$ for $(h,k)=(1,1),(1,2),(2,3)$ by the trapezoidal rule (step $0.05$), and compare with the two main terms of Theorem 7.14.

**P7.6 (★) Huge or tiny.** Exercise 7.1 says that $P_T(\frac12\le|\zeta(\frac12+it)|\le2)\to0$. Estimate it at $T=10^6$ and $T=10^{10}$ from the samples of P7.2, and compare with the Gaussian prediction $2\Phi(\log2/\sqrt{v})-1$ for both variances $v$.

## Solutions

### P7.1

In [ ]:
def theta(t):
    return t / 2 * np.log(t / (2 * np.pi)) - t / 2 - np.pi / 8 + 1 / (48 * t) + 7 / (5760 * t ** 3)

def Z_rs(t):
    t = np.asarray(t, dtype=float)
    a = np.sqrt(t / (2 * np.pi)); N = np.floor(a).astype(int); p = a - N
    th = theta(t); S = np.zeros_like(t)
    for n in range(1, N.max() + 1):
        m = n <= N
        S[m] += np.cos(th[m] - t[m] * math.log(n)) / math.sqrt(n)
    c = np.cos(2 * np.pi * p); c = np.where(np.abs(c) < 1e-9, 1e-9, c)
    return 2 * S + (-1.0) ** (N - 1) * a ** (-0.5) * np.cos(2 * np.pi * (p * p - p - 1 / 16)) / c

for t in [1000.37, 54321.123, 1e6 + 0.5]:
    print(f"t={t}: Riemann-Siegel {Z_rs([t])[0]:.7f}   mpmath {float(mpmath.siegelz(t)):.7f}")
ts = rng.uniform(1e6, 1e6 + 100, 10**4)
t0 = time.time(); Z_rs(ts); print(f"10^4 points near 10^6: {time.time() - t0:.2f} s")

The error is of order $t^{-3/4}$ (about $3\cdot10^{-5}$ at $t=10^6$, up to a constant). Each point costs $\sqrt{t/2\pi}\approx400$ cosines at $t=10^6$, against about $t$ terms for a direct Dirichlet series, and numpy processes thousands of points at once. Near a zero the relative error of $\log|Z|$ is large, but only on a set of tiny measure.

### P7.2

In [ ]:
samples = {}
for T, n in [(1e6, 20000), (1e10, 5000)]:
    ts = rng.uniform(T, 2 * T, n)
    L = np.log(np.abs(np.concatenate([Z_rs(ts[i:i + 500]) for i in range(0, n, 500)])))
    samples[T] = L
    v1 = 0.5 * math.log(math.log(T)); Nn = math.log(1.5 * T / (2 * math.pi)); v2 = 0.5 * (math.log(Nn) + 1 + 0.5772156649)
    print(f"T={T:.0e}: mean {L.mean():.3f}, variance {L.var():.3f};  (1/2)loglogT = {v1:.3f},  RMT {v2:.3f}")
    print("   KS vs N(0,(1/2)loglogT):", kstest(L / math.sqrt(v1), 'norm').statistic.round(3),
          "  KS vs N(0,RMT):", kstest(L / math.sqrt(v2), 'norm').statistic.round(3))

The mean is close to $0$, as it should be. Selberg's variance is far too small at these heights, while the random matrix variance matches the sample variance within a few per cent. Yet the Kolmogorov–Smirnov distance is about $0.05$ against both Gaussians: the law is not yet Gaussian in shape, with a long left tail coming from the points near zeros (compare Figure 7.1). The theorem is true, but its error term, a constant in the variance divided by $\log\log T$, is not small for any $T$ we can reach: $\log\log10^{10}\approx3.1$.

### P7.3

In [ ]:
T = 1e6; s0 = 0.5 + 1 / math.log(T); X = 100
ps = primes_upto(X).astype(float)
ts = rng.uniform(T, 2 * T, 10**5)
ReP0 = np.zeros_like(ts)
for p in ps:
    ReP0 += np.cos(ts * math.log(p)) / p ** s0
v = np.sum(ps ** (-2 * s0))
print(f"v = {v:.3f}")
for k, g in zip(range(1, 7), [0, 1, 0, 3, 0, 15]):
    print(f"k={k}: normalised moment {np.mean(ReP0 ** k) / (v / 2) ** (k / 2):7.3f}   Gaussian {g}")

The low moments match well, but the sixth moment falls short of $15$. The proof explains both: by Lemma 7.5 the moments of $\operatorname{Re}P_0$ are exactly those of the model $\sum_p\frac{\operatorname{Re}X_p}{p^{\sigma_0}}$ (Exercise 7.2) up to $X^{2k}/T$, which is tiny for $k\le3$ here. What separates the model from a Gaussian is the term $O(v^{k/2-2})$ from repeated primes, and with $v\approx1.4$ that is not small: the model is a sum of only $25$ independent terms, dominated by $p=2,3,5$. The convergence to the Gaussian is in $v\to\infty$, and $v$ grows like $\log\log T$.

### P7.4

In [ ]:
mpmath.mp.dps = 15
T = 1e6; LT = math.log(T)
ts = rng.uniform(T, 2 * T, 400)
Lh = np.log(np.abs(Z_rs(ts)))
for c in [0.25, 0.5, 1, 2, 4]:
    s = 0.5 + c / LT
    Ls = np.array([float(mpmath.log(abs(mpmath.zeta(mpmath.mpc(s, t))))) for t in ts])
    d = np.mean(np.abs(Lh - Ls))
    print(f"c={c}: mean |difference| = {d:.3f},  ratio to c = {d / c:.3f}")

The mean difference grows with $c$ much more slowly than linearly: it increases by about $0.15$ each time $c$ doubles, like $\log c$ rather than $c$, and the ratio to $c$ decreases. Proposition 7.3 is an upper bound, of the right order only for small $c$; for larger $c$ the difference reflects the extra fluctuation of $\log|\zeta|$ between the two lines, which the Bohr–Jessen variance $\frac12\log\frac1{\sigma-\frac12}$ suggests is logarithmic in $c$. The proof needs only that the difference be $o(\sqrt{\log\log T})$, and the crude linear bound is enough.

### P7.5

In [ ]:
T = 1000.0; sigma = 0.6; step = 0.05
tg = np.arange(T, 2 * T + step / 2, step)
z2 = np.array([abs(complex(mpmath.zeta(mpmath.mpc(sigma, t)))) ** 2 for t in tg])
w = np.full(len(tg), step); w[0] = w[-1] = step / 2
J = np.sum(w * (tg / (2 * math.pi)) ** (1 - 2 * sigma))
for h, k in [(1, 1), (1, 2), (2, 3)]:
    integral = np.sum(w * (h / k) ** (1j * tg) * z2)
    A = math.gcd(h, k) ** 2 / (h * k)
    main = float(mpmath.zeta(2 * sigma)) * A ** sigma * T + float(mpmath.zeta(2 - 2 * sigma)) * A ** (1 - sigma) * J
    print(f"(h,k)=({h},{k}): integral {integral.real:9.2f} {integral.imag:+8.2f}i   main terms {main:9.2f}")

For $(1,1)$ the two main terms (the second is negative, since $\zeta(0.8)<0$) match the integral to within $0.1\%$ (the bound $T^{1-\sigma}(\log T)^3$ of Theorem 7.14 is not small at $T=1000$; the true error is much smaller). For $(1,2)$ and $(2,3)$ the twist $(h/k)^{it}$ makes the integral oscillate, yet its real part still follows the main terms to about $1\%$, and the imaginary part stays small. The main terms decrease with $A=\frac{(h,k)^2}{hk}$: the mean square of $\zeta$ remembers the arithmetic of $h$ and $k$. This is what the mollifier exploits.

### P7.6

In [ ]:
for T, L in samples.items():
    emp = np.mean(np.abs(L) <= math.log(2))
    v1 = 0.5 * math.log(math.log(T)); Nn = math.log(1.5 * T / (2 * math.pi)); v2 = 0.5 * (math.log(Nn) + 1 + 0.5772156649)
    pred = lambda v: 2 * norm.cdf(math.log(2) / math.sqrt(v)) - 1
    print(f"T={T:.0e}: empirical {emp:.3f},  Gaussian with (1/2)loglogT {pred(v1):.3f},  with RMT variance {pred(v2):.3f}")

About $40\%$ of the values of $|\zeta(\frac12+it)|$ still lie between $\frac12$ and $2$ at these heights, and the limit $0$ is approached like $1/\sqrt{\log\log T}$: at $T=10^{100}$ the Gaussian prediction is still about $0.3$. 'Typically huge or tiny' is a theorem about very large $t$.